**Minor Project 2**

Name: Srushti  
Project: SpendDNA  

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("rahul_transactions.csv")

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Dataset loaded successfully!
Rows: 1328
Columns: 8


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [3]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nTransaction types:")
print(df["Type"].value_counts(dropna=False))

Column names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   object 
 1   Time         1328 non-null   object 
 2   Description  1328 non-null   object 
 3   Type         1328 non-null   object 
 4   Amount       1328 non-null   object 
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   object 
 7   Ref          1328 non-null   object 
dtypes: float64(1), object(7)
memory usage: 83.1+ KB

Missing values:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64

Transaction types:
Type
DR       670
Debit    652
CR         6
Name: count, dtype: int64


In [4]:
# Make a copy
df = df.copy()

# Count duplicates before removing them
duplicate_count = df.duplicated().sum()

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Convert Date
df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

# Clean Amount
df["amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

# Standardize transaction type
df["type"] = (
    df["Type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "dr": "debit",
        "debit": "debit",
        "cr": "credit",
        "credit": "credit"
    })
)

print("Original rows:", 1328)
print("Duplicates removed:", duplicate_count)
print("Rows after cleaning:", len(df))

print("\nUnparseable dates:", df["date"].isna().sum())
print("Unparseable amounts:", df["amount"].isna().sum())

print("\nData types:")
print(df[["date", "amount", "type"]].dtypes)

Original rows: 1328
Duplicates removed: 18
Rows after cleaning: 1310

Unparseable dates: 0
Unparseable amounts: 0

Data types:
date      datetime64[ns]
amount           float64
type              object
dtype: object


In [5]:
format="mixed"

In [6]:
print("Starting date:", df["date"].min().date())
print("Ending date:", df["date"].max().date())

Starting date: 2024-01-01
Ending date: 2024-06-30


In [7]:
vendor_keywords = {

    "Swiggy": ["SWIGGY", "BUNDL"],

    "Zomato": ["ZOMATO"],

    "Zepto": ["ZEPTO", "KIRANAKART"],

    "Blinkit": ["BLINKIT", "GROFERS"],

    "Amazon Prime": [
        "AMAZON PRIME",
        "AMZN PRIME",
        "UPI-AMAZON-PRIME"
    ],

    "Amazon": [
        "AMAZON",
        "AMZN",
        "AMAZONPAY"
    ],

    "Flipkart": ["FLIPKART", "FKART"],

    "Myntra": ["MYNTRA"],

    "Nykaa": ["NYKAA", "FSN E-COMMERCE"],

    "DMart": [
        "DMART",
        "AVENUE SUPERMARTS",
        "INNOVATIVE RETAIL"
    ],

    "BigBasket": ["BIGBASKET"],

    "Instamart": ["INSTAMART"],

    "Groww": [
        "GROWW",
        "NEXTBILLION"
    ],

    "Zerodha": ["ZERODHA"],

    "Ola": [
        "OLA",
        "ANI TECHNOLOGIES"
    ],

    "Uber": ["UBER"],

    "Rapido": ["RAPIDO"],

    "BMTC": [
        "BMTC",
        "TUMMOC"
    ],

    "Starbucks": [
        "STARBUCKS",
        "TATA STARBUCKS"
    ],

    "Cafe Coffee Day": [
        "COFFEE DAY",
        "CCD"
    ],

    "Third Wave Coffee": [
        "THIRD WAVE",
        "THIRDWAVE",
        "TWC INDIA"
    ],

    "Truffles": ["TRUFFLES"],

    "Meghana Foods": ["MEGHANA"],

    "Empire Restaurant": [
        "EMPIRE RESTAURANT"
    ],

    "Dineout": ["DINEOUT"],

    "Restaurant": ["RESTAURANT"],

    "Netflix": ["NETFLIX"],

    "Spotify": ["SPOTIFY"],

    "Disney Hotstar": [
        "HOTSTAR",
        "DISNEY HOTSTAR",
        "STAR INDIA"
    ],

    "JioFiber": ["JIOFIBER"],

    "Airtel": [
        "AIRTEL",
        "BHARTI AIRTEL"
    ],

    "Jio": [
        "RELIANCE JIO",
        "JIORECHARGE"
    ],

    "Vodafone Idea": [
        "VODAFONE IDEA",
        "VI POSTPAID",
        "VI-RECHARGE"
    ],

    "BESCOM": [
        "BESCOM",
        "BANGALORE ELEC"
    ],

    "BWSSB": ["BWSSB"],

    "Indian Oil": [
        "INDIAN OIL",
        "IOC"
    ],

    "BPCL": ["BPCL"],

    "HP": ["HP PETROL"],

    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS",
        "BIGTREE"
    ],

    "Transport": [
        "ROPPEN TRANSPORTATION"
    ]
}

In [8]:
vendor_order = list(vendor_keywords.keys())


def extract_vendor(description):

    text = str(description).upper()

    # ATM withdrawal
    if text.startswith("ATM-WDL"):
        return "Cash Withdrawal"

    # Salary
    if "SALARY" in text:
        return "Salary"

    # Rent
    if "RENT-LANDLORD" in text:
        return "Rent"

    # Check P2P transfers
    merchant_keywords = []

    for values in vendor_keywords.values():
        merchant_keywords.extend(values)

    if (
        text.startswith("UPI-")
        and "@" in text
        and not any(keyword in text for keyword in merchant_keywords)
    ):
        return "P2P Transfer"

    # Vendor matching
    for vendor in vendor_order:

        keywords = vendor_keywords[vendor]

        for keyword in keywords:

            if keyword in text:
                return vendor

    return "Uncategorised"

In [9]:
df["vendor_clean"] = df["Description"].apply(extract_vendor)

print("Number of canonical vendors:",
      df["vendor_clean"].nunique())

print("\nTop vendors:")
print(df["vendor_clean"].value_counts().head(15))

Number of canonical vendors: 44

Top vendors:
vendor_clean
Swiggy               223
Zomato               121
Ola                   87
Amazon                76
Zepto                 71
Uber                  71
Blinkit               55
Flipkart              47
Starbucks             42
Rapido                41
BMTC                  37
Third Wave Coffee     31
DMart                 30
Restaurant            29
Cafe Coffee Day       26
Name: count, dtype: int64


In [10]:
uncategorised = df[
    df["vendor_clean"] == "Uncategorised"
]

print("Uncategorised transactions:",
      len(uncategorised))

print("\nDescriptions not mapped:")
print(
    uncategorised["Description"]
    .unique()
)

Uncategorised transactions: 0

Descriptions not mapped:
[]


In [11]:
category_map = {

    # Food
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick commerce
    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",

    # Groceries
    "BigBasket": "Groceries",
    "DMart": "Groceries",

    # E-commerce
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    # Subscriptions
    "Amazon Prime": "Subscriptions",
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney Hotstar": "Subscriptions",

    # Transport
    "Ola": "Transport",
    "Uber": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",
    "Transport": "Transport",

    # Cafe
    "Starbucks": "Cafe",
    "Cafe Coffee Day": "Cafe",
    "Third Wave Coffee": "Cafe",

    # Restaurants
    "Restaurant": "Restaurants",
    "Truffles": "Restaurants",
    "Meghana Foods": "Restaurants",
    "Empire Restaurant": "Restaurants",
    "Dineout": "Restaurants",

    # Utilities
    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vodafone Idea": "Utilities",
    "JioFiber": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",

    # Investments
    "Groww": "Investments",
    "Zerodha": "Investments",

    # Fuel
    "Indian Oil": "Fuel",
    "BPCL": "Fuel",
    "HP": "Fuel",

    # Entertainment
    "BookMyShow": "Entertainment",

    # Special cases
    "P2P Transfer": "Personal Transfer",
    "Rent": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",
    "Salary": "Income"
}

In [12]:
df["category"] = (
    df["vendor_clean"]
    .map(category_map)
    .fillna("Uncategorised")
)

In [13]:
print(df["category"].value_counts())

category
Food Delivery        344
Transport            250
E-commerce           162
Quick Commerce       146
Cafe                  99
Restaurants           73
Utilities             43
Subscriptions         41
Groceries             41
Fuel                  28
Personal Transfer     24
Investments           23
Cash Withdrawal       17
Entertainment         13
Income                 6
Name: count, dtype: int64


In [14]:
debits = df[df["type"] == "debit"].copy()
credits = df[df["type"] == "credit"].copy()

print("Debit transactions:", len(debits))
print("Credit transactions:", len(credits))

Debit transactions: 1304
Credit transactions: 6


In [15]:
debits = df[df["type"] == "debit"].copy()
credits = df[df["type"] == "credit"].copy()

print("Debit transactions:", len(debits))
print("Credit transactions:", len(credits))

Debit transactions: 1304
Credit transactions: 6


In [16]:
total_credits = credits["amount"].sum()
total_debits = debits["amount"].sum()

net_change = total_credits - total_debits

savings_rate = (
    net_change / total_credits * 100
)

print("=" * 60)
print("SPENDING OVERVIEW")
print("=" * 60)

print(f"Total Credits : ₹{total_credits:,.2f}")
print(f"Total Debits  : ₹{total_debits:,.2f}")
print(f"Net Change    : ₹{net_change:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")
print(f"Transactions  : {len(df)}")
print(f"Unique Vendors: {df['vendor_clean'].nunique()}")

SPENDING OVERVIEW
Total Credits : ₹509,774.00
Total Debits  : ₹1,678,901.00
Net Change    : ₹-1,169,127.00
Savings Rate  : -229.34%
Transactions  : 1310
Unique Vendors: 44


In [17]:
spending_df = debits[
    ~debits["category"].isin([
        "Personal Transfer"
    ])
].copy()

category_spend = (
    spending_df
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

print("\nTOP CATEGORIES")
print("=" * 60)

for category, amount in category_spend.head(10).items():

    percentage = (
        amount / spending_df["amount"].sum() * 100
    )

    print(
        f"{category:<20} "
        f"{percentage:>6.2f}%   "
        f"₹{amount:>12,.2f}"
    )


TOP CATEGORIES
E-commerce            38.40%   ₹  593,769.00
Investments           16.05%   ₹  248,160.00
Food Delivery          9.75%   ₹  150,839.00
Restaurants            7.61%   ₹  117,737.00
Fuel                   5.78%   ₹   89,303.00
Quick Commerce         4.78%   ₹   73,882.00
Groceries              3.84%   ₹   59,407.00
Transport              3.72%   ₹   57,474.00
Cash Withdrawal        2.94%   ₹   45,500.00
Utilities              2.71%   ₹   41,914.00


In [21]:
top_vendors = (
    spending_df
    .groupby("vendor_clean")["amount"]
    .agg(["sum", "count"])
    .sort_values("sum", ascending=False)
    .head(5)
)

print("\nTOP 5 VENDORS")
print("=" * 60)

for vendor, row in top_vendors.iterrows():

    print(
        f"{vendor:<20} "
        f"₹{row['sum']:>12,.2f} "
        f"({int(row['count'])} transactions)"
    )


TOP 5 VENDORS
Amazon               ₹  318,422.00 (76 transactions)
Zerodha              ₹  210,000.00 (14 transactions)
Flipkart             ₹  177,510.00 (47 transactions)
Swiggy               ₹   95,523.00 (223 transactions)
Myntra               ₹   69,529.00 (20 transactions)


In [22]:
df["month"] = df["date"].dt.month

df["month_name"] = df["date"].dt.strftime("%b")

In [27]:
df["month"] = df["date"].dt.month

In [28]:
# Create month column in spending_df
spending_df["month"] = spending_df["date"].dt.month

# Create month names
spending_df["month_name"] = spending_df["date"].dt.strftime("%b")

# Check
print(spending_df[["date", "month", "month_name"]].head())

        date  month month_name
0 2024-01-01      1        Jan
1 2024-01-01      1        Jan
4 2024-01-01      1        Jan
5 2024-01-01      1        Jan
6 2024-01-01      1        Jan


In [29]:
monthly_spending = spending_df.pivot_table(
    values="amount",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

print(monthly_spending)

month                  1        2         3        4        5         6
category                                                               
Cafe              3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Cash Withdrawal   2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce       97134.0  92773.0  103772.0  68098.0  95001.0  136991.0
Entertainment     1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery    22633.0  23740.0   24803.0  27756.0  25408.0   26499.0
Fuel             30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries        17649.0   8571.0    6289.0  11748.0   9718.0    5432.0
Investments      38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce   11054.0  15177.0   14026.0  11870.0  11947.0    9808.0
Restaurants      16320.0  21772.0   28313.0   7711.0  22286.0   21335.0
Subscriptions     4256.0   5866.0    7952.0   3289.0   2619.0    4597.0
Transport        11005.0  10191.0    6857.0   9284.0  13141.0   

In [30]:
month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}

monthly_spending = monthly_spending.rename(
    columns=month_names
)

print("\nMONTHLY SPENDING")
print(monthly_spending)


MONTHLY SPENDING
month                Jan      Feb       Mar      Apr      May       Jun
category                                                               
Cafe              3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Cash Withdrawal   2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce       97134.0  92773.0  103772.0  68098.0  95001.0  136991.0
Entertainment     1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery    22633.0  23740.0   24803.0  27756.0  25408.0   26499.0
Fuel             30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries        17649.0   8571.0    6289.0  11748.0   9718.0    5432.0
Investments      38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce   11054.0  15177.0   14026.0  11870.0  11947.0    9808.0
Restaurants      16320.0  21772.0   28313.0   7711.0  22286.0   21335.0
Subscriptions     4256.0   5866.0    7952.0   3289.0   2619.0    4597.0
Transport        11005.0  10191.0    6857.0   

In [31]:
growth_data = []

for category in monthly_spending.index:

    first_month = monthly_spending.loc[category, "Jan"]
    last_month = monthly_spending.loc[category, "Jun"]

    if first_month > 0:

        growth = (
            (last_month - first_month)
            / first_month
            * 100
        )

        growth_data.append(
            (category, growth)
        )

growth_data = sorted(
    growth_data,
    key=lambda x: x[1],
    reverse=True
)

print("BIGGEST MONTH-TO-MONTH GROWTH")
print(growth_data[0])

BIGGEST MONTH-TO-MONTH GROWTH
('Cash Withdrawal', np.float64(750.0))


In [32]:
print("\nBIGGEST DECLINE")
print(growth_data[-1])


BIGGEST DECLINE
('Fuel', np.float64(-90.49534991095575))


In [33]:
df["hour"] = (
    df["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

In [37]:
columns="hour"

In [38]:
# Create time-related columns
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")

# Extract hour from Time column
df["hour"] = df["Time"].astype(str).str[:2].astype(int)

# Create spending dataframe
spending_df = df[
    (df["type"] == "debit") &
    (~df["category"].isin(["Personal Transfer"]))
].copy()

# Check that hour was created
print(spending_df[["Time", "hour"]].head())

    Time  hour
0  03:11     3
1  05:44     5
4  14:23    14
5  14:48    14
6  14:50    14


In [39]:
time_matrix = pd.pivot_table(
    spending_df,
    values="amount",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)

print("TIME-OF-DAY SPENDING")
print(time_matrix)

TIME-OF-DAY SPENDING
hour                  0        1        2        3        4        5   \
category                                                                
Cafe               872.0    549.0      0.0    316.0      0.0      0.0   
Cash Withdrawal      0.0      0.0      0.0      0.0      0.0      0.0   
E-commerce       18650.0   9801.0  10246.0  13554.0   9738.0  13305.0   
Entertainment      562.0    620.0      0.0      0.0      0.0      0.0   
Food Delivery      832.0   3017.0    928.0   1702.0   2618.0   3488.0   
Fuel              2675.0   2048.0      0.0    676.0   2010.0   2105.0   
Groceries         3895.0   1536.0   2352.0  10064.0   1169.0      0.0   
Investments       4883.0  15000.0      0.0      0.0  18834.0   4496.0   
Quick Commerce    1542.0   1402.0   1143.0   1165.0    768.0   1448.0   
Restaurants       1411.0    960.0   1336.0   1714.0      0.0   2126.0   
Subscriptions        0.0   1038.0   1295.0   5340.0   6965.0   4698.0   
Transport          706.0    26

In [40]:
def get_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

spending_df["time_period"] = spending_df["hour"].apply(get_time_period)

time_summary = spending_df.groupby("time_period")["amount"].sum()

print("TIME-OF-DAY SPENDING")
print(time_summary)

TIME-OF-DAY SPENDING
time_period
Afternoon    371806.0
Evening      355591.0
Morning      462376.0
Night        356529.0
Name: amount, dtype: float64


In [41]:
print("\nTIME-OF-DAY SPENDING")
print("=" * 60)

hourly_total = (
    spending_df
    .groupby("hour")["amount"]
    .sum()
)

max_spend = hourly_total.max()

for hour in range(24):

    value = hourly_total.get(hour, 0)

    bar_length = int(
        (value / max_spend) * 30
    ) if max_spend > 0 else 0

    print(
        f"{hour:02d}:00 | "
        f"{'#' * bar_length:<30} "
        f"₹{value:,.0f}"
    )


TIME-OF-DAY SPENDING
00:00 | ########                       ₹37,969
01:00 | ########                       ₹38,882
02:00 | ####                           ₹20,283
03:00 | #######                        ₹34,868
04:00 | #########                      ₹43,901
05:00 | #######                        ₹33,671
06:00 | ###########                    ₹55,506
07:00 | ####                           ₹19,938
08:00 | ########                       ₹41,070
09:00 | #################              ₹84,276
10:00 | ############################## ₹140,526
11:00 | ##################             ₹87,389
12:00 | ################               ₹75,236
13:00 | ############                   ₹57,505
14:00 | #####################          ₹101,182
15:00 | ########                       ₹39,647
16:00 | ####################           ₹98,236
17:00 | ###########                    ₹53,018
18:00 | #####################          ₹100,479
19:00 | ##############                 ₹66,553
20:00 | ###########################

In [42]:
food_delivery = df[
    df["category"] == "Food Delivery"
].copy()

late_night_food = food_delivery[
    (food_delivery["hour"] >= 21) |
    (food_delivery["hour"] <= 2)
]

late_night_percentage = (
    len(late_night_food)
    / len(food_delivery)
    * 100
)

print(
    f"Late-night Food Delivery: "
    f"{late_night_percentage:.2f}%"
)

Late-night Food Delivery: 21.22%


In [43]:
category_mean = (
    df.groupby("category")["amount"]
    .transform("mean")
)

category_std = (
    df.groupby("category")["amount"]
    .transform("std")
)

df["z_score"] = (
    (df["amount"] - category_mean)
    / category_std
)

In [44]:
anomalies = df[
    (df["type"] == "debit") &
    (df["z_score"] > 2)
].copy()

anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)

print("Number of anomalies:", len(anomalies))

Number of anomalies: 29


In [45]:
print("\nTOP 5 ANOMALIES")
print("=" * 80)

for _, row in anomalies.head(5).iterrows():

    print(
        f"{row['date'].strftime('%d %b')} | "
        f"{row['vendor_clean']:<20} | "
        f"{row['category']:<18} | "
        f"₹{row['amount']:>10,.2f} | "
        f"z = {row['z_score']:.2f}"
    )


TOP 5 ANOMALIES
26 Jun | Amazon               | E-commerce         | ₹ 22,008.00 | z = 3.97
07 Feb | Amazon               | E-commerce         | ₹ 21,986.00 | z = 3.97
26 Feb | Restaurant           | Restaurants        | ₹  8,383.00 | z = 3.88
22 Jun | Dineout              | Restaurants        | ₹  7,935.00 | z = 3.63
31 Mar | Meghana Foods        | Restaurants        | ₹  7,931.00 | z = 3.63


In [46]:
def detect_foodie(spending, total):

    food_amount = spending[
        spending["category"].isin([
            "Food Delivery",
            "Restaurants",
            "Cafe"
        ])
    ]["amount"].sum()

    percentage = food_amount / total * 100

    return percentage > 25, percentage

In [47]:
def detect_quick_commerce(spending, total):

    amount = spending[
        spending["category"] == "Quick Commerce"
    ]["amount"].sum()

    percentage = amount / total * 100

    return percentage > 15, percentage

In [48]:
def detect_shopaholic(spending, total):

    amount = spending[
        spending["category"] == "E-commerce"
    ]["amount"].sum()

    percentage = amount / total * 100

    return percentage > 15, percentage

In [49]:
def detect_investor(spending, total):

    amount = spending[
        spending["category"] == "Investments"
    ]["amount"].sum()

    percentage = amount / total * 100

    return percentage > 15, percentage

In [50]:
def detect_late_night_snacker(data):

    food = data[
        data["category"] == "Food Delivery"
    ]

    if len(food) == 0:
        return False, 0

    late = food[
        (food["hour"] >= 21) |
        (food["hour"] <= 2)
    ]

    percentage = len(late) / len(food) * 100

    return percentage > 50, percentage

In [51]:
def detect_subscription_lover(data):

    subscriptions = data[
        data["category"] == "Subscriptions"
    ]

    vendor_count = (
        subscriptions["vendor_clean"]
        .nunique()
    )

    return vendor_count >= 5, vendor_count

In [52]:
def detect_yolo(total_credits, total_debits):

    savings_rate = (
        (total_credits - total_debits)
        / total_credits
        * 100
    )

    return savings_rate < 10, savings_rate

In [53]:
def detect_disciplined_saver(
    total_credits,
    total_debits
):

    savings_rate = (
        (total_credits - total_debits)
        / total_credits
        * 100
    )

    return savings_rate > 40, savings_rate

In [54]:
print("\nRAHUL'S SPENDING ARCHETYPES")
print("=" * 60)

total_spending = spending_df["amount"].sum()

foodie, food_pct = detect_foodie(
    spending_df,
    total_spending
)

if foodie:
    print(
        f"THE FOODIE "
        f"({food_pct:.2f}% on food)"
    )


quick, quick_pct = detect_quick_commerce(
    spending_df,
    total_spending
)

if quick:
    print(
        f"THE QUICK COMMERCE JUNKIE "
        f"({quick_pct:.2f}% on quick commerce)"
    )


shopaholic, shop_pct = detect_shopaholic(
    spending_df,
    total_spending
)

if shopaholic:
    print(
        f"THE SHOPAHOLIC "
        f"({shop_pct:.2f}% on e-commerce)"
    )


investor, invest_pct = detect_investor(
    spending_df,
    total_spending
)

if investor:
    print(
        f"THE INVESTOR "
        f"({invest_pct:.2f}% on investments)"
    )


late, late_pct = detect_late_night_snacker(df)

if late:
    print(
        f"THE LATE-NIGHT SNACKER "
        f"({late_pct:.2f}% late-night food)"
    )


subscription, sub_count = detect_subscription_lover(df)

if subscription:
    print(
        f"THE SUBSCRIPTION LOVER "
        f"({sub_count} vendors)"
    )


yolo, yolo_rate = detect_yolo(
    total_credits,
    total_debits
)

if yolo:
    print(
        f"THE YOLO SPENDER "
        f"(savings rate {yolo_rate:.2f}%)"
    )


disciplined, disciplined_rate = (
    detect_disciplined_saver(
        total_credits,
        total_debits
    )
)

if disciplined:
    print(
        f"THE DISCIPLINED SAVER "
        f"(savings rate {disciplined_rate:.2f}%)"
    )


RAHUL'S SPENDING ARCHETYPES
THE SHOPAHOLIC (38.40% on e-commerce)
THE INVESTOR (16.05% on investments)
THE YOLO SPENDER (savings rate -229.34%)


final report

In [55]:
print()
print("=" * 70)
print("                 SpendDNA REPORT")
print("                    RAHUL SHARMA")
print("=" * 70)

print(
    f"6 months | {len(df)} transactions | "
    f"Jan-Jun 2024"
)

print("\nEXECUTIVE SUMMARY")
print("-" * 70)

print(f"Total Credits : ₹{total_credits:,.2f}")
print(f"Total Debits  : ₹{total_debits:,.2f}")
print(f"Net Change    : ₹{net_change:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")
print(f"Unique Vendors: {df['vendor_clean'].nunique()}")

print("\nTOP CATEGORIES")
print("-" * 70)

total_spend = spending_df["amount"].sum()

for category, amount in category_spend.head(5).items():

    pct = amount / total_spend * 100

    bars = "#" * int(pct / 2)

    print(
        f"{category:<20} "
        f"{bars:<25} "
        f"{pct:>6.2f}% "
        f"₹{amount:,.0f}"
    )

print("\nTOP VENDORS")
print("-" * 70)

for vendor, row in top_vendors.iterrows():

    print(
        f"{vendor:<20} "
        f"₹{row['sum']:>10,.0f} "
        f"({int(row['count'])} transactions)"
    )

print("\nTOP ANOMALIES")
print("-" * 70)

for _, row in anomalies.head(5).iterrows():

    print(
        f"{row['date'].strftime('%d %b')} | "
        f"{row['vendor_clean']:<18} | "
        f"₹{row['amount']:>9,.0f} | "
        f"z={row['z_score']:.2f}"
    )

print("\nTIME-OF-DAY")
print("-" * 70)

print(
    f"Late-night Food Delivery: "
    f"{late_pct:.2f}%"
)

print("\n" + "=" * 70)
print("                    END OF REPORT")
print("=" * 70)


                 SpendDNA REPORT
                    RAHUL SHARMA
6 months | 1310 transactions | Jan-Jun 2024

EXECUTIVE SUMMARY
----------------------------------------------------------------------
Total Credits : ₹509,774.00
Total Debits  : ₹1,678,901.00
Net Change    : ₹-1,169,127.00
Savings Rate  : -229.34%
Unique Vendors: 44

TOP CATEGORIES
----------------------------------------------------------------------
E-commerce           ###################        38.40% ₹593,769
Investments          ########                   16.05% ₹248,160
Food Delivery        ####                        9.75% ₹150,839
Restaurants          ###                         7.61% ₹117,737
Fuel                 ##                          5.78% ₹89,303

TOP VENDORS
----------------------------------------------------------------------
Amazon               ₹   318,422 (76 transactions)
Zerodha              ₹   210,000 (14 transactions)
Flipkart             ₹   177,510 (47 transactions)
Swiggy               ₹ 